In [1]:
# Install only the packages used by this optimized baseline.
# In a persistent runtime, you only need to run this once.
!pip install -q "transformers>=4.48,<5" accelerate langchain-core pydantic requests pandas rapidfuzz sacrebleu bert-score==0.3.13 sentencepiece


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 9.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-

In [2]:
# Import the core Python and evaluation dependencies.
import gc, os, re, time, unicodedata, torch, pandas as pd
from pathlib import Path
from collections import Counter
from pydantic import BaseModel
from typing import List
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score


In [3]:
# Import only the model/tokenizer plus PromptTemplate.
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_core.prompts import PromptTemplate


In [4]:
# Define the same agriculture question-answer data schema.
class QNAPair(BaseModel):
    id: int
    question: str
    reference_answer: str

class AgriData(BaseModel):
    qa_pairs: List[QNAPair]


In [5]:
# Load the same 200-question test set from the GitHub Data folder.
import requests
TEST_URL = "https://raw.githubusercontent.com/RamijWasithRahat/Bangla-Agriculture-Chatbot/main/Data/Bangla_Agriculture_QA_Test_200.json"
data = requests.get(TEST_URL, timeout=60).json()
agri_data = AgriData(**data)
print("Test examples:", len(agri_data.qa_pairs))


Test examples: 200


In [6]:
# Load the Mistral 7B Instruct v0.3 baseline model and tokenizer.
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Left padding is required for efficient batched decoder-only generation.
# It has no effect when batch size is 1.
tokenizer.padding_side = "left"

dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else (torch.float16 if torch.cuda.is_available() else torch.float32)
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
model.eval()

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GiB):", round(props.total_memory / 1024**3, 1))
else:
    print("WARNING: CUDA GPU not detected. A 7B model on CPU will be very slow.")

if hasattr(model, "hf_device_map"):
    print("Device map:", model.hf_device_map)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

GPU: Tesla T4
GPU memory (GiB): 14.6
Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 0, 'model.layers.17': 0, 'model.layers.18': 0, 'model.layers.19': 0, 'model.layers.20': 0, 'model.layers.21': 0, 'model.layers.22': 0, 'model.layers.23': 0, 'model.layers.24': 0, 'model.layers.25': 0, 'model.layers.26': 0, 'model.layers.27': 0, 'model.layers.28': 0, 'model.layers.29': 0, 'model.layers.30': 'cpu', 'model.layers.31': 'cpu', 'model.norm': 'cpu', 'model.rotary_emb': 'cpu', 'lm_head': 'cpu'}


In [7]:
# Use the Mistral EOS token as the generation terminator.
terminators = [tokenizer.eos_token_id]


In [8]:
# Keep the same deterministic generation settings, but choose a safe batch size.
MAX_NEW_TOKENS = 100

def choose_generation_batch_size():
    if not torch.cuda.is_available():
        return 1
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if total_gb >= 70:
        return 16
    if total_gb >= 38:
        return 8
    if total_gb >= 22:
        return 4
    if total_gb >= 15:
        return 2
    return 1

GEN_BATCH_SIZE = choose_generation_batch_size()
print("Initial generation batch size:", GEN_BATCH_SIZE)


Initial generation batch size: 1


In [9]:
# Define the original zero-shot prompt.
zero_shot_prompt = PromptTemplate(
    template="""নিচের প্রশ্নটির উত্তর বাংলায় দিন।\n\nপ্রশ্নঃ {question}\nউত্তরঃ""",
    input_variables=["question"],
)


In [10]:
# Define the original one-shot prompt.
one_shot_prompt = PromptTemplate(
    template="""নিচের উদাহরণটি লক্ষ্য করুন এবং একইভাবে বাংলায় উত্তর দিন।\n\nপ্রশ্নঃ টমেটো গাছে পার্শ্বকুশি ও মরা পাতা ছাঁটাই করার সুবিধা কী?\nউত্তরঃ পার্শ্বকুশি ও মরা পাতা ছাঁটাই করলে গাছে রোগ-বালাই ও\nপোকার আক্রমণ কমে এবং ফলের আকার বড় হয়।\n\nপ্রশ্নঃ {question}\nউত্তরঃ""",
    input_variables=["question"],
)


In [11]:
# Define the original few-shot prompt.
few_shot_prompt = PromptTemplate(
    template="""নিচের উদাহরণগুলো লক্ষ্য করুন এবং একইভাবে বাংলায় উত্তর দিন।\n\nপ্রশ্নঃ টমেটোর চারা রোপণের সঠিক দূরত্ব ও বয়স কত?\nউত্তরঃ ২৫-৩০ দিন বয়সের টমেটোর চারা প্রতিটি বেডে দুটি\nসারিতে ৬০×৪০ সেমি দূরত্বে রোপণ করতে হয়।\n\nপ্রশ্নঃ টমেটো চাষে হেক্টরপ্রতি কী পরিমাণ গোবর সার প্রয়োগ করতে হয়?\nউত্তরঃ টমেটোর ভালো ফলনের জন্য হেক্টরপ্রতি ৮ থেকে ১২ টন\nপচা গোবর সার প্রয়োগ করতে হয়।\n\nপ্রশ্নঃ টমেটোর জমিতে ইউরিয়া এবং এমপি সার কখন ও কীভাবে দিতে হয়?\nউত্তরঃ পার্শ্বকুশি ছাঁটাইয়ের পর চারা লাগানোর ৩য় ও ৫ম\nসপ্তাহে রিং পদ্ধতিতে দুই কিস্তিতে এই সারগুলো দিতে হয়।\n\nপ্রশ্নঃ টমেটো গাছে পার্শ্বকুশি ও মরা পাতা ছাঁটাই করার সুবিধা কী?\nউত্তরঃ পার্শ্বকুশি ও মরা পাতা ছাঁটাই করলে গাছে রোগ-বালাই ও\nপোকার আক্রমণ কমে এবং ফলের আকার বড় হয়।\n\nপ্রশ্নঃ টমেটো গাছকে প্রবল বাতাস থেকে রক্ষা করতে কী ব্যবস্থা নেওয়া হয়?\nউত্তরঃ টমেটো গাছ যাতে নুয়ে না পড়ে সেজন্য বাঁশের তৈরি কাঠি,\nকঞ্চি বা ডাল দিয়ে ‘এ’ আকৃতির ঠেকনা দেওয়া হয়।\n\nপ্রশ্নঃ {question}\nউত্তরঃ""",
    input_variables=["question"],
)


In [12]:
# Define the original chain-of-thought-style prompt.
cot_prompt = PromptTemplate(
    template="""আপনি একজন কৃষি বিশেষজ্ঞ।\n\nপ্রশ্নটি বুঝে প্রয়োজনীয় তথ্য বিবেচনা করুন।\nসবশেষে শুধু সংক্ষিপ্ত ও সঠিক চূড়ান্ত উত্তর বাংলায় দিন।\n\nপ্রশ্নঃ {question}\n\nচূড়ান্ত উত্তরঃ""",
    input_variables=["question"],
)


In [13]:
# Define the original instruction prompt.
instruction_prompt = PromptTemplate(
    template="""আপনি একজন কৃষি সহায়ক বাংলা প্রশ্নোত্তর সহকারী।\n\nনির্দেশনা:\n- শুধুমাত্র বাংলায় উত্তর দিন\n- সংক্ষিপ্ত ও নির্ভুল উত্তর দিন\n- অতিরিক্ত ব্যাখ্যা যোগ করবেন না\n- প্রশ্ন অনুযায়ী সরাসরি উত্তর দিন\n\nপ্রশ্নঃ {question}\nউত্তরঃ""",
    input_variables=["question"],
)


In [14]:
# Register all five original baseline prompting methods.
PROMPTS = {
    "zero_shot": zero_shot_prompt,
    "one_shot": one_shot_prompt,
    "few_shot": few_shot_prompt,
    "chain_of_thought": cot_prompt,
    "instruction": instruction_prompt,
}


In [15]:
# Build exactly the same chat-template text used by the baseline.
SYSTEM_TEXT = "আপনি একজন কৃষি বিষয়ক প্রশ্নোত্তর সহকারী। আপনাকে বাংলায় সংক্ষিপ্ত ও সঠিক উত্তর দিতে হবে।"

def render_model_prompt(prompt_template, question):
    user_prompt = prompt_template.format(question=question)
    messages = [
        {"role": "system", "content": SYSTEM_TEXT},
        {"role": "user", "content": user_prompt},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


In [16]:
# Create the same model-specific output folder, but persist it in Google Drive.
# This is what makes the notebook resumable after a Colab runtime disconnect.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PERSIST_ROOT = Path("/content/drive/MyDrive/Bangla_Agriculture_Baseline_Resumable")
    print("Persistent storage: Google Drive")
except Exception as exc:
    # This fallback is useful outside Colab, but a local folder will NOT survive
    # a Colab runtime reset.
    PERSIST_ROOT = Path("./Bangla_Agriculture_Baseline_Resumable")
    print("Google Drive was not mounted; using local storage:", exc)

BASELINE_OUT = PERSIST_ROOT / "baseline_results_mistral_7b_v0_3"
CHECKPOINT_DIR = BASELINE_OUT / "_checkpoints"

BASELINE_OUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

method_predictions = {}

def atomic_save_csv(df, path):
    """Safely replace a checkpoint so an interrupted write does not destroy the previous one."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(path.name + ".tmp")
    df.to_csv(temp_path, index=False, encoding="utf-8-sig")
    try:
        os.replace(temp_path, path)
    except OSError:
        # Google Drive/FUSE fallback.
        df.to_csv(path, index=False, encoding="utf-8-sig")
        if temp_path.exists():
            temp_path.unlink()

def read_csv_checkpoint(path):
    """Read saved text exactly; empty predictions should stay empty strings, not NaN."""
    return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False)

def validate_generation_checkpoint(df):
    """Make sure a saved checkpoint belongs to this exact 200-question dataset/order."""
    required = {"id", "question", "gold", "prediction", "truncated"}
    if not required.issubset(df.columns):
        return False

    if len(df) > len(agri_data.qa_pairs):
        return False

    for i, row in df.iterrows():
        pair = agri_data.qa_pairs[i]
        if str(row["id"]) != str(pair.id):
            return False
        if str(row["question"]) != str(pair.question):
            return False
        if str(row["gold"]) != str(pair.reference_answer):
            return False

    return True

print("Results folder:", BASELINE_OUT)
print("Checkpoint folder:", CHECKPOINT_DIR)


Mounted at /content/drive
Persistent storage: Google Drive
Results folder: /content/drive/MyDrive/Bangla_Agriculture_Baseline_Resumable/baseline_results_mistral_7b_v0_3
Checkpoint folder: /content/drive/MyDrive/Bangla_Agriculture_Baseline_Resumable/baseline_results_mistral_7b_v0_3/_checkpoints


In [17]:
# Generate one prompting method in batches, resuming from its last saved batch.
# Generation/model/prompt logic is the same as the previous FAST notebook.
def generate_method_predictions(prompt_template, method):
    checkpoint_path = CHECKPOINT_DIR / f"{method}_generation_checkpoint.csv"

    rows = []
    if checkpoint_path.exists():
        saved = read_csv_checkpoint(checkpoint_path)
        if not validate_generation_checkpoint(saved):
            raise ValueError(
                f"Checkpoint mismatch: {checkpoint_path}\n"
                "The saved checkpoint does not match the current test dataset/order."
            )
        rows = saved.to_dict("records")
        print(f"Resuming {method}: {len(rows)}/{len(agri_data.qa_pairs)} rows already saved")

    total = len(agri_data.qa_pairs)
    if len(rows) == total:
        print(f"{method}: generation already complete; skipping model generation.")
        return pd.DataFrame(rows)

    batch_size = GEN_BATCH_SIZE
    i = len(rows)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started = time.perf_counter()

    while i < total:
        current_pairs = agri_data.qa_pairs[i:i + batch_size]
        current_prompts = [
            render_model_prompt(prompt_template, pair.question)
            for pair in current_pairs
        ]

        try:
            encoded = tokenizer(
                current_prompts,
                return_tensors="pt",
                padding=True,
            )
            encoded = {k: v.to(model.device) for k, v in encoded.items()}
            input_width = encoded["input_ids"].shape[1]

            with torch.inference_mode():
                generated = model.generate(
                    **encoded,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=terminators,
                )

            new_token_ids = generated[:, input_width:]
            batch_predictions = tokenizer.batch_decode(
                new_token_ids,
                skip_special_tokens=True,
            )

            for pair, prediction in zip(current_pairs, batch_predictions):
                prediction = prediction.strip()
                token_count = len(
                    tokenizer(prediction, add_special_tokens=False)["input_ids"]
                )
                rows.append(
                    {
                        "id": pair.id,
                        "question": pair.question,
                        "gold": pair.reference_answer,
                        "prediction": prediction,
                        "truncated": token_count >= MAX_NEW_TOKENS,
                    }
                )

            i = len(rows)

            # Persistent checkpoint after EVERY successful generation batch.
            atomic_save_csv(pd.DataFrame(rows), checkpoint_path)
            print(f"{method}: checkpoint saved {i}/{total}")

        except torch.cuda.OutOfMemoryError:
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            torch.cuda.empty_cache()
            print(
                f"GPU memory was insufficient; reducing generation "
                f"batch size to {batch_size}"
            )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    result = pd.DataFrame(rows)
    atomic_save_csv(result, checkpoint_path)

    print(
        f"{method}: {len(result)}/{total} answers complete "
        f"in this session after {elapsed:.1f}s; final batch size {batch_size}"
    )
    return result


In [18]:
# Generate and retain predictions for all five prompting methods.
# A completed/partially completed method is restored automatically from Drive.
for method, prompt_template in PROMPTS.items():
    print(f"\nGenerating: {method}")
    method_predictions[method] = generate_method_predictions(
        prompt_template,
        method,
    )
    print(method, "available rows:", len(method_predictions[method]))

# No more 7B generation is needed below.
# Release its GPU memory so BERTScore can use the GPU safely.
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\nReleased generation-model memory before evaluation.")



Generating: zero_shot
Resuming zero_shot: 200/200 rows already saved
zero_shot: generation already complete; skipping model generation.
zero_shot available rows: 200

Generating: one_shot
Resuming one_shot: 200/200 rows already saved
one_shot: generation already complete; skipping model generation.
one_shot available rows: 200

Generating: few_shot
Resuming few_shot: 200/200 rows already saved
few_shot: generation already complete; skipping model generation.
few_shot available rows: 200

Generating: chain_of_thought
Resuming chain_of_thought: 200/200 rows already saved
chain_of_thought: generation already complete; skipping model generation.
chain_of_thought available rows: 200

Generating: instruction
Resuming instruction: 156/200 rows already saved
instruction: checkpoint saved 157/200
instruction: checkpoint saved 158/200
instruction: checkpoint saved 159/200
instruction: checkpoint saved 160/200
instruction: checkpoint saved 161/200
instruction: checkpoint saved 162/200
instructio

In [19]:
# Use the RAG notebook text normalization for every evaluation.
BN_TO_EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
def normalize(text):
    text = unicodedata.normalize("NFKC", str(text)).translate(BN_TO_EN).lower()
    text = re.sub(r"[^\u0980-\u09FFA-Za-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def tokens(text):
    return normalize(text).split()


In [20]:
# Use the same Exact Match and Token F1 definitions.
def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))

def token_f1(pred, gold):
    p, g = tokens(pred), tokens(gold)
    if not p or not g: return 0.0
    overlap = sum((Counter(p) & Counter(g)).values())
    if overlap == 0: return 0.0
    precision, recall = overlap / len(p), overlap / len(g)
    return 2 * precision * recall / (precision + recall)


In [21]:
# Use the same ROUGE-1 and ROUGE-2 definitions.
def rouge_n(pred, gold, n):
    p, g = tokens(pred), tokens(gold)
    if len(p) < n or len(g) < n: return 0.0
    pg = Counter(tuple(p[i:i+n]) for i in range(len(p)-n+1))
    gg = Counter(tuple(g[i:i+n]) for i in range(len(g)-n+1))
    overlap = sum((pg & gg).values())
    if overlap == 0: return 0.0
    precision, recall = overlap / sum(pg.values()), overlap / sum(gg.values())
    return 2 * precision * recall / (precision + recall)


In [22]:
# Use the same ROUGE-L definition.
def rouge_l(pred, gold):
    p, g = tokens(pred), tokens(gold)
    if not p or not g: return 0.0
    dp = [0] * (len(g) + 1)
    for x in p:
        new = [0]
        for j, y in enumerate(g, 1):
            new.append(dp[j-1] + 1 if x == y else max(dp[j], new[-1]))
        dp = new
    precision, recall = dp[-1] / len(p), dp[-1] / len(g)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


In [23]:
# Add the same lexical evaluation metrics to each prediction row.
def add_lexical_metrics(df):
    df = df.copy().fillna("")
    pairs = list(zip(df["prediction"], df["gold"]))
    df["Exact Match"] = [exact_match(p, g) for p, g in pairs]
    df["Fuzzy Match"] = [fuzz.token_set_ratio(normalize(p), normalize(g)) / 100 for p, g in pairs]
    df["Token F1"] = [token_f1(p, g) for p, g in pairs]
    df["ROUGE-1"] = [rouge_n(p, g, 1) for p, g in pairs]
    df["ROUGE-2"] = [rouge_n(p, g, 2) for p, g in pairs]
    df["ROUGE-L"] = [rouge_l(p, g) for p, g in pairs]
    return df


In [24]:
# Run the same multilingual BERTScore configuration.
# The optimized workflow still calls BERTScore once for all five methods together.
BERT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BERT_BATCH_SIZE = 64 if torch.cuda.is_available() else 8

def add_bert_metrics(df):
    df = df.copy()
    current_batch_size = BERT_BATCH_SIZE

    while True:
        try:
            started = time.perf_counter()
            P, R, F1 = bert_score(
                df["prediction"].astype(str).tolist(),
                df["gold"].astype(str).tolist(),
                model_type="bert-base-multilingual-cased",
                batch_size=current_batch_size,
                device=BERT_DEVICE,
                idf=False,
                rescale_with_baseline=False,
                verbose=True,
            )
            elapsed = time.perf_counter() - started
            break

        except torch.cuda.OutOfMemoryError:
            if current_batch_size == 1:
                raise
            current_batch_size = max(1, current_batch_size // 2)
            torch.cuda.empty_cache()
            print(
                "BERTScore GPU memory was insufficient; "
                f"retrying with batch size {current_batch_size}"
            )

    df["BERT Precision"] = P.cpu().numpy()
    df["BERT Recall"] = R.cpu().numpy()
    df["BERT F1"] = F1.cpu().numpy()

    print(
        f"BERTScore completed for {len(df)} pairs on {BERT_DEVICE} "
        f"in {elapsed:.1f}s using batch size {current_batch_size}"
    )
    return df


In [25]:
# Summarize the same corpus BLEU and aggregate metric values.
def summarize_metrics(df):
    bleu = BLEU(tokenize="none", smooth_method="exp", effective_order=True)
    pred_texts = [" ".join(tokens(x)) for x in df["prediction"]]
    gold_texts = [" ".join(tokens(x)) for x in df["gold"]]
    corpus_bleu = bleu.corpus_score(pred_texts, [gold_texts]).score / 100
    names = ["Exact Match", "Fuzzy Match", "Corpus BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "Token F1", "BERT Precision", "BERT Recall", "BERT F1", "Truncated Outputs"]
    values = [df["Exact Match"].mean(), df["Fuzzy Match"].mean(), corpus_bleu, df["ROUGE-1"].mean(), df["ROUGE-2"].mean(), df["ROUGE-L"].mean(), df["Token F1"].mean(), df["BERT Precision"].mean(), df["BERT Recall"].mean(), df["BERT F1"].mean(), df["truncated"].astype(str).str.lower().eq("true").sum()]
    return pd.DataFrame({"metric": names, "score": values})


In [26]:
# Lexical scoring is unchanged.
# BERTScore is intentionally applied later to the combined 1,000-row table.
def evaluate_lexical_metrics(df):
    return add_lexical_metrics(df)


In [27]:
# Evaluate with persistent stage checkpoints.
# Metric definitions and final output format are unchanged.
expected_rows = len(agri_data.qa_pairs) * len(PROMPTS)
combined_scored_path = CHECKPOINT_DIR / "combined_scored_with_bertscore.csv"
combined_lexical_path = CHECKPOINT_DIR / "combined_lexical_metrics.csv"

# Highest-level resume point: if BERTScore already finished, load it directly.
if combined_scored_path.exists():
    candidate = read_csv_checkpoint(combined_scored_path)
    bert_cols = {"BERT Precision", "BERT Recall", "BERT F1", "method"}

    if len(candidate) == expected_rows and bert_cols.issubset(candidate.columns):
        combined_scored = candidate
        print(
            f"Loaded completed BERTScore stage: "
            f"{len(combined_scored)}/{expected_rows} rows"
        )
    else:
        print("Existing combined BERTScore checkpoint is incomplete; rebuilding it.")
        combined_scored = None
else:
    combined_scored = None

if combined_scored is None:
    combined_parts = []

    # Resume lexical evaluation independently for each prompting method.
    for method, pred_df in method_predictions.items():
        method_lexical_path = CHECKPOINT_DIR / f"{method}_lexical_metrics.csv"

        if method_lexical_path.exists():
            scored = read_csv_checkpoint(method_lexical_path)
            needed_cols = {
                "Exact Match", "Fuzzy Match", "Token F1",
                "ROUGE-1", "ROUGE-2", "ROUGE-L"
            }

            if len(scored) == len(pred_df) and needed_cols.issubset(scored.columns):
                print(f"{method}: loaded completed lexical metrics")
            else:
                scored = evaluate_lexical_metrics(pred_df)
                atomic_save_csv(scored, method_lexical_path)
                print(f"{method}: rebuilt and checkpointed lexical metrics")
        else:
            scored = evaluate_lexical_metrics(pred_df)
            atomic_save_csv(scored, method_lexical_path)
            print(f"{method}: checkpointed lexical metrics")

        scored = scored.copy()
        scored.insert(0, "method", method)
        combined_parts.append(scored)

    combined_lexical = pd.concat(combined_parts, ignore_index=True)
    atomic_save_csv(combined_lexical, combined_lexical_path)
    print("Combined lexical stage checkpointed.")

    # Same BERTScore calculation as before.
    # If the runtime disconnects during this call, the lexical stage is already safe.
    combined_scored = add_bert_metrics(combined_lexical)
    atomic_save_csv(combined_scored, combined_scored_path)
    print("Combined BERTScore stage checkpointed.")

# Save the same per-method final prediction/result CSV files.
all_results = []
for method in PROMPTS:
    scored_df = (
        combined_scored.loc[combined_scored["method"] == method]
        .drop(columns=["method"])
        .reset_index(drop=True)
    )

    result_df = summarize_metrics(scored_df)

    atomic_save_csv(
        scored_df,
        BASELINE_OUT / f"{method}_predictions.csv",
    )
    atomic_save_csv(
        result_df,
        BASELINE_OUT / f"{method}_result.csv",
    )

    temp = result_df.copy()
    temp.insert(0, "method", method)
    all_results.append(temp)
    print("Saved:", method)


zero_shot: checkpointed lexical metrics
one_shot: checkpointed lexical metrics
few_shot: checkpointed lexical metrics
chain_of_thought: checkpointed lexical metrics
instruction: checkpointed lexical metrics
Combined lexical stage checkpointed.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/17 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/16 [00:00<?, ?it/s]

done in 3.48 seconds, 287.21 sentences/sec
BERTScore completed for 1000 pairs on cuda in 19.8s using batch size 64
Combined BERTScore stage checkpointed.
Saved: zero_shot
Saved: one_shot
Saved: few_shot
Saved: chain_of_thought
Saved: instruction


In [28]:
# Save one combined CSV to compare all baseline prompting methods.
baseline_comparison = pd.concat(all_results, ignore_index=True)
atomic_save_csv(
    baseline_comparison,
    BASELINE_OUT / "baseline_all_methods_result.csv",
)

display(
    baseline_comparison.pivot(
        index="metric",
        columns="method",
        values="score",
    )
)
print("Saved persistently to:", BASELINE_OUT)
print(
    "If Colab disconnects later, reconnect and Run all; "
    "saved stages in _checkpoints will be reused."
)


method,chain_of_thought,few_shot,instruction,one_shot,zero_shot
metric,,,,,
BERT F1,0.697184,0.673903,0.705406,0.705335,0.694966
BERT Precision,0.692845,0.667331,0.706861,0.703827,0.690902
BERT Recall,0.702440,0.681400,0.704981,0.707760,0.699935
Corpus BLEU,0.022050,0.024824,0.021899,0.029142,0.020111
Exact Match,0.000000,0.000000,0.000000,0.000000,0.000000
Fuzzy Match,0.480845,0.464323,0.492676,0.494755,0.483900
ROUGE-1,0.155903,0.136891,0.164716,0.169772,0.153795
ROUGE-2,0.043348,0.046532,0.041284,0.051095,0.043621
ROUGE-L,0.139083,0.125254,0.144550,0.152541,0.136327


Saved persistently to: /content/drive/MyDrive/Bangla_Agriculture_Baseline_Resumable/baseline_results_mistral_7b_v0_3
If Colab disconnects later, reconnect and Run all; saved stages in _checkpoints will be reused.
